# PySCF adapter: certify a Hartree-Fock state

Walks the PySCF adapter end-to-end on a tiny molecule (H2 in STO-3G). The mean-field Hartree-Fock state in its canonical molecular-orbital basis sits in the Bernoulli class, so the certificate is exact and zero.

Practical takeaway: any chemistry workflow that starts from canonical-basis Hartree-Fock has a zero-cost zero-bias certificate for every word in the chemistry catalog as a free starting point. Correlation introduces nonzero $\Delta$; bound it via the UCB diagnostic or by direct cumulant evaluation on the correlated state.

In [ ]:
%pip install -q "cumulant_residual_cert[pyscf]@git+https://github.com/kootru-repo/cumulant-residual-cert.git"

In [ ]:
from pyscf import gto, scf

mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g', verbose=0)
mf = scf.RHF(mol).run(conv_tol=1e-10)
print(f'Converged: {mf.converged}')
print(f'E(HF) = {mf.e_tot:.10f} Hartree')

In [ ]:
from cumulant_residual_cert import Catalog
from cumulant_residual_cert.adapters.pyscf import from_mean_field

cat = Catalog.chemistry_r4()
est = from_mean_field(mf, cat)

print(f'Delta              = {est.delta}')
print(f'Delta_is_exact     = {est.delta_is_exact}')
print(f'Framework          = {est.framework}')
print(f'Level              = {est.bound.level}')
print()
print('Notes:')
for note in est.notes:
    print(f'  - {note}')
print()
print('Certified bias bars:')
for word, bar in est.bound.bounds.items():
    print(f'  |tau({word:32s})| <= {bar}')

## What this is and is not

**Is:** A rigorous deterministic bound on the bias of a length-$\le 4$ catalog observable when reconstructed via order-$\le 2$ cumulant truncation from this state.

**Is not:** A measurement-advantage claim. The cost of measuring 1- and 2-RDM on hardware (the inputs to the cumulant reconstruction) is unchanged.

**Beyond Hartree-Fock.** For post-HF states (CISD, CCSD, CASCI, FCI), $\Delta$ is nonzero in general. Three routes:
1. Compute $\Delta$ analytically from RDMs (when feasible).
2. Estimate an upper bound on $\Delta$ via shadow tomography (`cumulant_residual_cert.delta_ucb`, or matchgate shadows via the OpenFermion adapter at $r \ge 3$).
3. Plug a target tolerance into `certify(catalog, delta=epsilon, ...)` and use the bound as a go/no-go signal on the workflow.